## Following illustration of Compressive Sensing according to:

## http://www.pyrunner.com/weblog/2016/05/26/compressed-sensing-python/m

In [1]:
# make sure you've got the following packages installed
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.optimize as spopt
import scipy.fftpack as spfft
import scipy.ndimage as spimg
import cvxpy as cvx

import matplotlib.pyplot as plt
import matplotlib

# showing figures inline
%matplotlib inline

# plotting options 
font = {'size'   : 20}
plt.rc('font', **font)
plt.rc('text', usetex=True)

matplotlib.rc('figure', figsize=(18, 6) )

ModuleNotFoundError: No module named 'cvxpy'

# Why is L1 Making Sense?

## First Showing both Optimizations

In [ ]:
# generate some data with noise
x = np.sort(np.random.uniform(0, 10, 15))
y = 3 + 0.2 * x + 0.1 * np.random.randn(len(x))

# find L1 line fit
l1_fit = lambda x0, x, y: np.sum(np.abs(x0[0] * x + x0[1] - y))
xopt1 = spopt.fmin(func=l1_fit, x0=[1, 1], args=(x, y))

# find L2 line fit
l2_fit = lambda x0, x, y: np.sum(np.power(x0[0] * x + x0[1] - y, 2))
xopt2 = spopt.fmin(func=l2_fit, x0=[1, 1], args=(x, y))

plt.plot( x, y, 'o', ms='12')
plt.plot( x, xopt1[0]*x + xopt1[1], linewidth=2.0, label='$L^1$')
plt.plot( x, xopt2[0]*x + xopt2[1], linewidth=2.0, label='$L^2$')
plt.legend(loc='lower right')

## Adding Outliers and Repeating Optimization

In [ ]:
# adjust data by adding outlyers
y2 = y.copy()
y2[3] += 4
y2[13] -= 3

# refit the lines
xopt12 = spopt.fmin(func=l1_fit, x0=[1, 1], args=(x, y2))
xopt22 = spopt.fmin(func=l2_fit, x0=[1, 1], args=(x, y2))


plt.plot( x, y2, 'o', ms='12')
plt.plot( x, xopt12[0]*x + xopt12[1], 'g', linewidth=2.0, label='$L^1$')
plt.plot( x, xopt22[0]*x + xopt22[1], 'b', linewidth=2.0, label='$L^2$')
plt.legend(loc='lower right')

# Now Let's Look at CS...

### First a Little Motivation

In [ ]:
# sum of two sinusoids
n = 5000

t = np.linspace(0, 1/8, n)

t_S = 1/8/n

y = np.sin(1394 * np.pi * t) + np.sin(3266 * np.pi * t)

Y = spfft.dct(y, norm='ortho')
f = np.linspace( -1/2/t_S, 1/2/t_S, n)

plt.figure()

plt.subplot(121)
plt.plot(t,y)
plt.xlabel('$t$')
plt.ylabel('$y(t)$')

plt.subplot(122)
plt.plot(t,y)
plt.xlim((0, .01))
plt.xlabel('$t$')
plt.ylabel('$y(t)$')

plt.figure()

plt.subplot(121)
plt.plot( Y)
plt.xlabel('$f$')
plt.ylabel('$Y(f)$')

plt.subplot(122)
plt.plot( Y)
plt.xlim( (0, 500 ) )
plt.xlabel('$f$')
plt.ylabel('$Y(f)$')

## Now Let's Use a Small Sub-Signal

In [ ]:
# extract small sample of signal
m = 200 # 10% sample

ri = np.random.choice(n, m, replace=False) # random sample of indices
ri.sort() # sorting not strictly necessary, but convenient for plotting

t2 = t[ri]
y2 = y[ri]

plt.figure()

plt.subplot(121)
plt.plot(t2,y2)
plt.xlabel('$t$')
plt.ylabel('$y_{CS}(t)$')

plt.subplot(122)
plt.plot(t2,y2)
plt.xlim((0, .01))
plt.xlabel('$t$')
plt.ylabel('$y_{CS}(t)$')

### Get Signal by Applying Optimization

In [ ]:
# create idct matrix operator
A = spfft.idct(np.identity(n), norm='ortho', axis=0)
A = A[ri]

# do L1 optimization by using according toolbox

# define n optimization variables
vx = cvx.Variable(n)

# objective is minimization of || . ||_1
# constraint is that samples are reproduced
objective = cvx.Minimize(cvx.norm(vx, 1))
constraints = [A*vx == y2]

# define and solve the problem
prob = cvx.Problem(objective, constraints)
result = prob.solve(verbose=False)

In [ ]:
# reconstruct signal
x = np.array(vx.value)
x = np.squeeze(x)

sig = spfft.idct(x, norm='ortho', axis=0)
SIG = spfft.dct( sig , norm='ortho' )


# plotting
plt.figure()

plt.subplot(121)
plt.plot(t, y)
plt.xlim( (0, .01) )
plt.xlabel('$t$')
plt.ylabel('$y(t)$')

plt.subplot(122)
plt.plot( Y )
plt.xlabel('$f$')
plt.ylabel('$Y(f)$')


plt.figure()

plt.subplot(121)
plt.plot(t, sig)
plt.xlim( (0, .01) )
plt.xlabel('$t$')
plt.ylabel('$\hat{y}(t)$')

plt.subplot(122)
plt.plot( SIG)
plt.xlabel('$f$')
plt.ylabel('$\hat{Y}(f)$')